In [32]:
from utils.random_forest_utils.rf_preprocessing_utils import WindowAlgPreprocessor

rf_preprocessor_clamping = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/clamping_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_clamping, target_df_clamping = rf_preprocessor_clamping.read_data()
sensors_df_clamping = rf_preprocessor_clamping.feature_selection()
rf_preprocessor_clamping.normalize_angle()
rf_preprocessor_bending = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/bending_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_bending, target_df_bending = rf_preprocessor_bending.read_data()
sensors_df_bending = rf_preprocessor_bending.feature_selection()
rf_preprocessor_bending.normalize_angle()
rf_preprocessor_declamping = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/declamping_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_declamping, target_df_declamping = rf_preprocessor_declamping.read_data()
sensors_df_declamping = rf_preprocessor_declamping.feature_selection()
rf_preprocessor_declamping.normalize_angle()

,Experiment_ID,Angle[degree]ORDistance[mm],Secondary-axis [mm],Main-axis [mm],Out-of-roundness [-],Collapse [mm]
0,2,0.000000,0.900079,0.313091,0.999954,0.313091
1,2,0.022017,0.908296,0.425584,0.834481,0.425584
2,2,0.044033,0.905549,0.591178,0.582994,0.591178
3,2,0.066050,0.901145,0.751356,0.338801,0.751356
4,2,0.088067,0.909703,0.867774,0.167582,0.867774
...,...,...,...,...,...,...
14558,318,0.902686,0.928583,0.029142,0.997804,0.029142
14559,318,0.924703,0.922565,0.025609,1.000000,0.025609
14560,318,0.946720,0.921322,0.032466,0.992607,0.032466
14561,318,0.968736,0.925369,0.051137,0.974276,0.051137


In [33]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis, entropy

def resample_experiment_fast(group, n=46, metric='mean'):
    """
    Optimized resampling function using vectorized operations.
    Up to 10-100x faster than the original implementation.
    """
    # Sort by time
    group = group.sort_values('Time_[s]')
    time_col = group['Time_[s]'].values
    
    # Assign each row to a time bin
    time_bins = np.linspace(time_col.min(), time_col.max(), n + 1)
    bin_indices = np.digitize(time_col, time_bins[:-1]) - 1
    bin_indices = np.clip(bin_indices, 0, n - 1)
    
    # Get experiment ID
    exp_id = group['Experiment_ID'].iloc[0]
    
    # Select numeric columns only
    cols_to_process = [col for col in group.columns 
                       if col not in ['Time_[s]', 'Experiment_ID']]
    
    results = []
    
    # Process each bin
    for bin_idx in range(n):
        mask = bin_indices == bin_idx
        if not mask.any():
            continue
            
        row_data = {'Experiment_ID': exp_id}
        
        for col in cols_to_process:
            values = group[col].values[mask]
            if len(values) == 0:
                continue
            
            # Compute metric using vectorized operations
            if metric == 'mean':
                row_data[f'{col}_mean'] = values.mean()
            elif metric == 'median':
                row_data[f'{col}_median'] = np.median(values)
            elif metric == 'min':
                row_data[f'{col}_min'] = values.min()
            elif metric == 'max':
                row_data[f'{col}_max'] = values.max()
            elif metric == 'range':
                row_data[f'{col}_range'] = values.ptp()
            elif metric == 'std':
                row_data[f'{col}_std'] = values.std()
            elif metric == 'var':
                row_data[f'{col}_var'] = values.var()
            elif metric == 'mad':
                row_data[f'{col}_mad'] = np.abs(values - values.mean()).mean()
            elif metric == 'rms':
                row_data[f'{col}_rms'] = np.sqrt((values ** 2).mean())
            elif metric == 'skew':
                row_data[f'{col}_skew'] = skew(values)
            elif metric == 'kurtosis':
                row_data[f'{col}_kurtosis'] = kurtosis(values)
            elif metric == 'energy':
                row_data[f'{col}_energy'] = (values ** 2).sum()
            elif metric == 'entropy':
                abs_vals = np.abs(values)
                probs = abs_vals / (abs_vals.sum() + 1e-12)
                row_data[f'{col}_entropy'] = entropy(probs + 1e-12)
            elif metric == 'cv':
                row_data[f'{col}_cv'] = values.std() / (values.mean() + 1e-12)
            elif metric == 'iqr':
                row_data[f'{col}_iqr'] = np.percentile(values, 75) - np.percentile(values, 25)
            elif metric == 'p25':
                row_data[f'{col}_p25'] = np.percentile(values, 25)
            elif metric == 'p75':
                row_data[f'{col}_p75'] = np.percentile(values, 75)
            elif metric == 'trend_slope':
                if len(values) > 1:
                    row_data[f'{col}_trend_slope'] = np.polyfit(np.arange(len(values)), values, 1)[0]
                else:
                    row_data[f'{col}_trend_slope'] = 0
        
        results.append(row_data)
    
    return pd.DataFrame(results)


# Alternative: Ultra-fast version using pandas groupby (even faster for 'mean', 'std', 'min', 'max')
def resample_experiment_ultrafast(group, n=46, metric='mean'):
    """
    Ultra-optimized version using pandas groupby operations.
    Works best for basic metrics like mean, std, min, max, median.
    """
    group = group.sort_values('Time_[s]')
    time_col = group['Time_[s]'].values
    
    # Assign bins
    time_bins = np.linspace(time_col.min(), time_col.max(), n + 1)
    group['_bin'] = np.digitize(time_col, time_bins[:-1]) - 1
    group['_bin'] = group['_bin'].clip(0, n - 1)
    
    # Select columns to aggregate
    cols_to_agg = [col for col in group.columns 
                   if col not in ['Time_[s]', 'Experiment_ID', '_bin']]
    
    # Map metric to pandas aggregation function
    agg_func_map = {
        'mean': 'mean',
        'median': 'median',
        'min': 'min',
        'max': 'max',
        'std': 'std',
        'var': 'var',
        'sum': 'sum'
    }
    
    if metric in agg_func_map:
        # Use fast pandas groupby
        result = group.groupby('_bin')[cols_to_agg].agg(agg_func_map[metric])
        result = result.add_suffix(f'_{metric}')
        result['Experiment_ID'] = group['Experiment_ID'].iloc[0]
        return result.reset_index(drop=True)
    else:
        # Fall back to custom implementation
        return resample_experiment_fast(group.drop('_bin', axis=1), n, metric)


# Usage - choose the best function for your needs:

# Option 2: Ultra-fast version (best for mean, std, min, max, median)
df_bending = sensors_df_bending.reset_index()
df_resampled_bending = (
    df_bending.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=46, metric='mean'))
    .reset_index(drop=True)
)

# Apply to all datasets
df_clamping = sensors_df_clamping.reset_index()
df_resampled_clamping = (
    df_clamping.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=46, metric='mean'))
    .reset_index(drop=True)
)

df_declamping = sensors_df_declamping.reset_index()
df_resampled_declamping = (
    df_declamping.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=46, metric='mean'))
    .reset_index(drop=True)
)

/tmp/ipykernel_97078/295691104.py:135: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: resample_experiment_ultrafast(g, n=46, metric='mean'))
/tmp/ipykernel_97078/295691104.py:143: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: resample_experiment_ultrafast(g, n=46, metric='mean'))
/tmp/ipykernel_97078/295691104.py:150: FutureWarning: DataFrameGroupBy.apply operated on the grouping colu

In [34]:
def normalize_experiment(group, n=46):
    if len(group) > n:
        # Just take the first 46 rows
        return group.iloc[:n].copy()
    else:
        # Already 46 rows
        return group.copy()

# Apply to each experiment
df_normalized_clamping = target_df_clamping.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=47)
df_normalized_clamping = df_normalized_clamping.reset_index(drop=True)
df_normalized_bending = target_df_bending.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=47)
df_normalized_bending = df_normalized_bending.reset_index(drop=True)
df_normalized_declamping = target_df_declamping.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=47)
df_normalized_declamping = df_normalized_declamping.reset_index(drop=True)

/tmp/ipykernel_97078/4038853369.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_normalized_clamping = target_df_clamping.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=47)
/tmp/ipykernel_97078/4038853369.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_normalized_bending = target_df_bending.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=47)
/tmp/ipyke

In [35]:
X_clamping = rf_preprocessor_clamping.group_and_pad(df_resampled_clamping, group_col="Experiment_ID")[55:,:,1:]
Y_clamping = rf_preprocessor_clamping.group_and_pad(df_normalized_clamping, group_col="Experiment_ID")[55:,:-1,1:]

X_bending = rf_preprocessor_bending.group_and_pad(df_resampled_bending, group_col="Experiment_ID")[55:,:,1:]
Y_bending = rf_preprocessor_bending.group_and_pad(df_normalized_bending, group_col="Experiment_ID")[55:,:-1,1:]

X_declamping = rf_preprocessor_declamping.group_and_pad(df_resampled_declamping, group_col="Experiment_ID")[:,:,1:]
Y_declamping = rf_preprocessor_declamping.group_and_pad(df_normalized_declamping, group_col="Experiment_ID")[55:,:-1,1:]

In [36]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import shap

# ---------------------------
# 1. Model fitting and evaluation
# ---------------------------
def fit_model(X, y, model_choice="random_forest"):
    # Only multi-output safe models
    models = {
        "random_forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "gradient_boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
        "linear_regression": LinearRegression(),
        "ridge": Ridge(alpha=1.0),
        "lasso": Lasso(alpha=0.1),
        "elasticnet": ElasticNet(alpha=0.1, l1_ratio=0.5),
        "mlp": MLPRegressor(hidden_layer_sizes=(100,100,100), random_state=42, max_iter=3000)
    }

    if model_choice not in models:
        print(f"Model {model_choice} not found. Using random_forest instead.")
        model_choice = "random_forest"

    model = models[model_choice]
    n_exp, n_timesteps, n_features = X.shape
    n_exp, n_timesteps, n_targets = y.shape
    X = X.reshape(n_exp * n_timesteps, n_features)  # shape: (100*46, 16)
    y = y.reshape(n_exp * n_timesteps, n_targets)
    # Ensure y is 2D
    y = np.asarray(y)
    if y.ndim == 1:
        y = y.reshape(-1, 1)

    # Fit model
    model.fit(X, y)

    # Predictions
    y_pred = model.predict(X)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)

    # Evaluate
    mae = mean_absolute_error(y, y_pred, multioutput="raw_values")
    r2 = r2_score(y, y_pred, multioutput="raw_values")
    
    for i, (mae_i, r2_i) in enumerate(zip(mae, r2)):
        print(f"Target {i+1}: MAE={mae_i:.4f}, R²={r2_i:.4f}")

    return model, y_pred

# ---------------------------
# 2. Compute feature window importance
# ---------------------------
def compute_feature_window_importance(X, y, model, window_size=20, stride=1, method_num=1):
    n_timesteps, n_features = X.shape
    n_feature_windows = (n_timesteps - window_size) // stride + 1
    feature_window_importance = []

    if method_num== 1:  # permutation importance
            for w in range(n_feature_windows):
                start = w * stride
                end = start + window_size
                if end > n_timesteps:
                    break

                X_perm = X.copy()
                for f in range(n_features):
                    X_perm[start:end, f] = np.random.permutation(X_perm[start:end, f])

                y_hat = model.predict(X_perm)
                imp = np.mean(np.abs(y - y_hat))
                feature_window_importance.append(imp)

    elif method_num== 2:  # mean decrease impurity (only for tree-based models)
            if hasattr(model, "feature_importances_"):
                mdi_importance = model.feature_importances_
                baseline_error = mean_absolute_error(y, model.predict(X))

                for w in range(n_feature_windows):
                    start = w * stride
                    end = start + window_size
                    if end > n_timesteps:
                        break

                    y_window = y[start:end]
                    y_pred_window = model.predict(X[start:end, :])
                    window_error = mean_absolute_error(y_window, y_pred_window)
                    window_importance = np.sum(mdi_importance) * (window_error / baseline_error)
                    feature_window_importance.append(window_importance)
            else:
                raise ValueError("MDI method requires a tree-based model with feature_importances_")
            
    elif method_num== 3:  # SHAP importance
            # Compute SHAP values
            explainer = shap.Explainer(model.predict, X)
            shap_values = explainer(X)
            
            # Calculate window importance as mean absolute SHAP values per window
            for w in range(n_feature_windows):
                start = w * stride
                end = start + window_size
                if end > n_timesteps:
                    break
                
                # Get SHAP values for this window and average across features and time
                window_shap = np.abs(shap_values.values[start:end, :])
                window_importance = np.mean(window_shap)
                feature_window_importance.append(window_importance)
                
    elif method_num== 4:  # leave-one-window-out importance
            baseline_pred = model.predict(X)
            baseline_error = np.mean(np.abs(y - baseline_pred))
            
            for w in range(n_feature_windows):
                start = w * stride
                end = start + window_size
                if end > n_timesteps:
                    break

                # Mask window with mean values
                X_masked = X.copy()
                window_means = np.mean(X[start:end, :], axis=0)
                X_masked[start:end, :] = window_means
                
                y_hat_masked = model.predict(X_masked)
                masked_error = np.mean(np.abs(y - y_hat_masked))
                imp = masked_error - baseline_error
                feature_window_importance.append(imp)
                
    elif method_num== 5:  # zero-out window importance
            baseline_pred = model.predict(X)
            baseline_error = np.mean(np.abs(y - baseline_pred))
            
            for w in range(n_feature_windows):
                start = w * stride
                end = start + window_size
                if end > n_timesteps:
                    break

                X_zero = X.copy()
                X_zero[start:end, :] = 0
                
                y_hat_zero = model.predict(X_zero)
                zero_error = np.mean(np.abs(y - y_hat_zero))
                imp = zero_error - baseline_error
                feature_window_importance.append(imp)

    elif method_num== 6:  # noise injection importance
            baseline_pred = model.predict(X)
            baseline_error = np.mean(np.abs(y - baseline_pred))
            
            for w in range(n_feature_windows):
                start = w * stride
                end = start + window_size
                if end > n_timesteps:
                    break

                X_noise = X.copy()
                noise = np.random.normal(0, 0.1, X_noise[start:end, :].shape)
                X_noise[start:end, :] += noise
                
                y_hat_noise = model.predict(X_noise)
                noise_error = np.mean(np.abs(y - y_hat_noise))
                imp = noise_error - baseline_error
                feature_window_importance.append(imp)

    elif method_num== 7:  # feature drop importance
            baseline_pred = model.predict(X)
            baseline_error = np.mean(np.abs(y - baseline_pred))
            
            for w in range(n_feature_windows):
                start = w * stride
                end = start + window_size
                if end > n_timesteps:
                    break

                X_drop = X.copy()
                X_drop[start:end, :] = np.mean(X_drop[start:end, :])
                
                y_hat_drop = model.predict(X_drop)
                drop_error = np.mean(np.abs(y - y_hat_drop))
                imp = drop_error - baseline_error
                feature_window_importance.append(imp)

    elif method_num== 8:  # partial dependence importance
            baseline_pred = model.predict(X)
            
            for w in range(n_feature_windows):
                start = w * stride
                end = start + window_size
                if end > n_timesteps:
                    break

                X_pd = X.copy()
                # Use random values from other parts of the data
                random_indices = np.random.choice(n_timesteps, end-start)
                X_pd[start:end, :] = X[random_indices, :]
                
                y_hat_pd = model.predict(X_pd)
                pd_change = np.mean(np.abs(baseline_pred - y_hat_pd))
                feature_window_importance.append(pd_change)

    elif method_num== 9:  # variance-based importance
            baseline_pred = model.predict(X)
            
            for w in range(n_feature_windows):
                start = w * stride
                end = start + window_size
                if end > n_timesteps:
                    break

                X_var = X.copy()
                # Scale the window to increase variance
                X_var[start:end, :] = X_var[start:end, :] * 2
                
                y_hat_var = model.predict(X_var)
                var_change = np.mean(np.abs(baseline_pred - y_hat_var))
                feature_window_importance.append(var_change)

    total_importance = np.sum(feature_window_importance)
    importance_percentages = (np.array(feature_window_importance) / total_importance) * 100
    return feature_window_importance, importance_percentages, n_feature_windows
            

# ---------------------------
# 3. Plotting
# ---------------------------
def plot_feature_windows(X, y, feature_window_importance, importance_percentages,
                         window_size=20, stride=1, num_top_windows=2, target_window_idx=0):
    
    n_timesteps, n_features = X.shape
    top_window_indices = np.argsort(feature_window_importance)[-num_top_windows:][::-1]
    target_start = target_window_idx * stride
    target_end = target_start + window_size
    timesteps = np.arange(n_timesteps)
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    colors = plt.cm.tab10(np.linspace(0, 1, n_features))
    
    # Plot features
    for f in range(n_features):
        axes[0].plot(timesteps, X[:, f], marker='.', markersize=5, color=colors[f],
                     linestyle='-', alpha=0.6, label=f'Feature {f+1}')
    
    # Highlight top windows
    highlight_colors = ['red', 'orange', 'purple', 'green', 'brown', 'cyan', 'magenta', 'yellow']
    for i, window_idx in enumerate(top_window_indices):
        if i < len(highlight_colors):
            start = window_idx * stride
            end = start + window_size
            axes[0].axvspan(start, end, color=highlight_colors[i], alpha=0.4,
                            label=f'Top {i+1} ({importance_percentages[window_idx]:.1f}%)')
            mid = (start + end) / 2
            y_range = axes[0].get_ylim()
            text_y = y_range[1] - 0.05 * (y_range[1] - y_range[0]) * (i+1)
            axes[0].text(mid, text_y, f'{importance_percentages[window_idx]:.1f}%',
                         ha='center', va='center', fontweight='bold',
                         bbox=dict(boxstyle="round,pad=0.3", facecolor=highlight_colors[i], alpha=0.8))
    
    axes[0].set_ylabel("Feature values")
    axes[0].set_title("Feature values with top important windows highlighted")
    axes[0].grid(True, alpha=0.3)
    
    # Plot target
    if y.ndim == 1:
        axes[1].plot(timesteps, y, marker='.', markersize=2, color='blue', linestyle='-', alpha=0.8, linewidth=0.5)
    else:
        for i in range(y.shape[1]):
            axes[1].plot(timesteps, y[:, i], marker='.', markersize=2, linestyle='-', alpha=0.8, label=f'Target {i+1}')
    
    axes[1].axvspan(target_start, target_end, color='grey', alpha=0.4, label='Selected target window')
    axes[1].set_xlabel("Time steps")
    axes[1].set_ylabel("Target")
    axes[1].set_title("Target values with selected window highlighted")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

# ---------------------------
# 4. Additional analysis plotting
# ---------------------------
def plot_importance_analysis(feature_window_importance, importance_percentages, num_top_windows=2):
    n_feature_windows = len(feature_window_importance)
    top_window_indices = np.argsort(feature_window_importance)[-num_top_windows:][::-1]
    highlight_colors = ['red', 'orange', 'purple', 'green', 'brown', 'cyan', 'magenta', 'yellow']

    plt.figure(figsize=(14, 8))
    
    # Importance scores
    plt.subplot(2, 2, 1)
    bars = plt.bar(range(n_feature_windows), feature_window_importance, color='skyblue', alpha=0.7)
    for i, window_idx in enumerate(top_window_indices):
        if i < len(highlight_colors):
            bars[window_idx].set_color(highlight_colors[i])
            bars[window_idx].set_alpha(0.8)
    plt.xlabel('Window Number')
    plt.ylabel('Importance Score')
    plt.title('Feature Window Importance Scores')
    plt.xticks(range(n_feature_windows))
    plt.grid(axis='y', alpha=0.3)

In [37]:
model_choice = 'random_forest'
model_clamping, y_pred_clamping = fit_model(X_clamping, Y_clamping, model_choice=model_choice)
model_bending, y_pred_bending = fit_model(X_bending, Y_bending, model_choice=model_choice)
model_declamping, y_pred_clamping = fit_model(X_declamping, Y_declamping, model_choice=model_choice)

Target 1: MAE=0.0368, R²=0.9692
Target 2: MAE=0.0325, R²=0.9660
Target 3: MAE=0.0377, R²=0.9591
Target 4: MAE=0.0325, R²=0.9660
Target 1: MAE=0.0355, R²=0.9701
Target 2: MAE=0.0203, R²=0.9822
Target 3: MAE=0.0257, R²=0.9773
Target 4: MAE=0.0203, R²=0.9822
Target 1: MAE=0.0338, R²=0.9723
Target 2: MAE=0.0204, R²=0.9848
Target 3: MAE=0.0234, R²=0.9832
Target 4: MAE=0.0204, R²=0.9848


In [38]:
import ipywidgets as widgets
from ipywidgets import interact

# ---------------------------
# Define available methods
# ---------------------------
method_options = {
    "1. Permutation Importance": 1,
    "2. Mean Decrease Impurity": 2,
    "3. SHAP Importance": 3,
    "4. Leave-One-Window-Out Importance": 4,
    "5. Zero-Out Window Importance": 5,
    "6. Noise Injection Importance": 6,
    "7. Feature Drop Importance": 7,
    "8. Partial Dependence Importance": 8,
    "9. Variance-Based Importance": 9
}

# ---------------------------
# Experiment selection dropdown
# ---------------------------
experiment_idx_all = sensors_df_clamping["Experiment_ID"].unique()[55:]  # assume all datasets share indices
experiment_idx_dropdown = widgets.Dropdown(
    options=experiment_idx_all,
    description="Experiment ID:"
)

# ---------------------------
# Sliders and dropdowns
# ---------------------------
window_size_slider = widgets.IntSlider(value=10, min=1, max=50, step=1, description='Window Size:')
stride_slider = widgets.IntSlider(value=10, min=1, max=50, step=1, description='Stride:')
method_dropdown = widgets.Dropdown(options=method_options, value=1, description='Method:')
num_top_windows_slider = widgets.IntSlider(value=2, min=1, max=10, step=1, description='Top Windows:')
target_window_idx_slider = widgets.IntSlider(value=2, min=0, max=10, step=1, description='Target Window:')

# ---------------------------
# Interactive function
# ---------------------------
@interact(
    experiment_idx=experiment_idx_dropdown,
    window_size=window_size_slider,
    stride=stride_slider,
    method_num=method_dropdown,
    num_top_windows=num_top_windows_slider,
    target_window_idx=target_window_idx_slider,
)
def interactive_feature_importance(experiment_idx, window_size, stride, method_num, num_top_windows, target_window_idx):
    
    experiment_idx = int(experiment_idx)
    
    # Find indices in each dataset
    idx_clamping = np.where(sensors_df_clamping["Experiment_ID"].unique()[55:] == experiment_idx)[0][0]
    idx_bending = np.where(sensors_df_bending["Experiment_ID"].unique()[55:] == experiment_idx)[0][0]
    idx_declamping = np.where(sensors_df_declamping["Experiment_ID"].unique()[:] == experiment_idx)[0][0]
    
    X_c, y_c = X_clamping[idx_clamping], Y_clamping[idx_clamping]
    X_b, y_b = X_bending[idx_bending], Y_bending[idx_bending]
    X_d, y_d = X_declamping[idx_declamping], Y_declamping[idx_declamping]
    
    # Compute feature window importance
    fwi_c, ip_c, _ = compute_feature_window_importance(X_c, y_c, model_clamping, window_size, stride, method_num)
    fwi_b, ip_b, _ = compute_feature_window_importance(X_b, y_b, model_bending, window_size, stride, method_num)
    fwi_d, ip_d, _ = compute_feature_window_importance(X_d, y_d, model_declamping, window_size, stride, method_num)
    
    # ---------------------------
    # Plot subplots
    # ---------------------------
    fig, axes = plt.subplots(3, 2, figsize=(18, 15))  # 3 rows, 2 columns
    
    def plot_features(ax, X, fwi, title):
        n_features = X.shape[1]
        colors = plt.cm.tab10(np.linspace(0, 1, n_features))
        for f in range(n_features):
            ax.plot(X[:, f], alpha=0.6, color=colors[f], label=f'Feature {f+1}')
        top_indices = np.argsort(fwi)[-num_top_windows:][::-1]
        highlight_colors = ['red', 'orange', 'purple', 'green', 'brown', 'cyan', 'magenta', 'yellow']
        for i, idx_w in enumerate(top_indices):
            start = idx_w * stride
            end = start + window_size
            ax.axvspan(start, end, color=highlight_colors[i], alpha=0.3)
        ax.set_title(title)
        ax.grid(True)
        # ax.legend()
    
    def plot_importance(ax, fwi, title):
        top_indices = np.argsort(fwi)[-num_top_windows:][::-1]
        highlight_colors = ['red', 'orange', 'purple', 'green', 'brown', 'cyan', 'magenta', 'yellow']
        ax.bar(range(len(fwi)), fwi, color='skyblue', alpha=0.7)
        for i, idx_w in enumerate(top_indices):
            ax.bar(idx_w, fwi[idx_w], color=highlight_colors[i], alpha=0.8)
        ax.set_title(title)
        ax.grid(True)
    
    # Clamping
    plot_features(axes[0,0], X_c, fwi_c, f"Clamping Features (Experiment {experiment_idx})")
    plot_importance(axes[0,1], fwi_c, "Clamping Importance")
    
    # Bending
    plot_features(axes[1,0], X_b, fwi_b, f"Bending Features (Experiment {experiment_idx})")
    plot_importance(axes[1,1], fwi_b, "Bending Importance")
    
    # Declamping
    plot_features(axes[2,0], X_d, fwi_d, f"Declamping Features (Experiment {experiment_idx})")
    plot_importance(axes[2,1], fwi_d, "Declamping Importance")
    
    plt.tight_layout()
    plt.show()


interactive(children=(Dropdown(description='Experiment ID:', options=(58, 59, 60, 61, 62, 63, 64, 65, 66, 67, …